# Named Entity Recognition with Encoders

<a href="https://colab.research.google.com/github/HassanAlgoz/dl/blob/main/modules/pipelines/02-pipelines/02_token_classification.ipynb" target="_blank">
  <img src="https://raw.githubusercontent.com/HassanAlgoz/dl/main/assets/Open%20in%20Colab-F9AB00.svg" alt="Open in Colab" height="50"/>
</a>

Document-level sentiment assigns one label per review; **token-level NER** identifies people, places, and organizations inside the text. This lab applies encoder pipelines to entity recognition.


**Goal:** Apply encoder models to Named Entity Recognition (NER).

**Topics:** default NER pipeline, subword vs whole-word entities, aggregation strategies.


In [1]:
# --- Setup: Clone repo & cd into correct folder (Colab only) ---
import os
import sys
import subprocess

if "google.colab" in sys.modules:
    repo_url = "https://github.com/HassanAlgoz/dl.git"
    lab_folder = "dl/modules/pipelines/02-pipelines"

    # Only clone if the folder doesn't exist
    if not os.path.exists(lab_folder):
        subprocess.run(["git", "clone", repo_url])

    # Change working directory to the lab folder
    os.chdir(lab_folder)


## Imports


In [ ]:
%pip install -qqq transformers


In [2]:
#imports
import numpy as np
import pandas as pd

from transformers import pipeline

# NLP Scenario
You are an analyst for a marketing company that just launched a new product suite of mobile devices. You have data from product reviews of the new product and need to identify the entities in the reviews.

#####  Product Reviews
1. "I absolutely love the TechWave X1! It has made my daily tasks in Spain so much easier and more efficient. Highly recommend it!"
2. "I'm not very impressed with the TechWave X1. It lacks some essential features and is quite slow here in Mexico."
3. "The TechWave X1 is fantastic! It has exceeded my expectations and has become an essential part of my daily routine."
4. "I found the TechWave X1 to be quite average. It does the job, but there's nothing particularly special about it."
5. "The TechWave X1 is terrible. It's full of glitches and crashes frequently. I regret purchasing it."
6.  "The TechWave X1 is disappointing. It doesn't live up to the hype and is missing several key functionalities."


In [3]:
# data setup
# create a list of reviews
reviews = ["I absolutely love the TechWave X1! It has made my daily tasks so much easier and more efficient. Highly recommend it!",
"I'm not very impressed with the TechWave X1. It lacks some essential features and is quite slow.",
"The TechWave X1 is fantastic! It has exceeded my expectations and has become an essential part of my daily routine.",
"I found the TechWave X1 to be quite average. It does the job, but there's nothing particularly special about it.",
"The TechWave X1 is terrible. It's full of glitches and crashes frequently. I regret purchasing it.",
"The TechWave X1 is disappointing. It doesn't live up to the hype and is missing several key functionalities."
]

# Initialize the pipeline for your task: Named Entity Recognition (NER)

The default analyzer for NER analysis can be called with:
```python
    pipeline("ner")
```
This pipeline can take data at sentence, paragraph, or document level and will return:
- entity- classification (see below for entity classification types)
- score- confidence of classification (range 0-1) where 1 is highly confident
- index- token position
- word - actual text, may be a subword (##subword)
- start/end- character position

All the models we use here have the encoder architecture. You could also consider an Encoder-Decoder model, but it is likely not needed for this task.

#### Entity Types:
- `O` means the word doesn’t correspond to any entity.
- `B-PER/I-PER` means the word corresponds to the beginning of/is inside a person entity.
- `B-ORG/I-ORG` means the word corresponds to the beginning of/is inside an organization entity.
- `B-LOC/I-LOC` means the word corresponds to the beginning of/is inside a location entity.
- `B-MISC/I-MISC` means the word corresponds to the beginning of/is inside a miscellaneous entity.  
source: [Hugging Face](https://huggingface.co/learn/nlp-course/en/chapter7/2?fw=pt)


In [11]:
# initalize the NER pipeline
ner_pipeline = pipeline(
  task="token-classification",
  model="dbmdz/bert-large-cased-finetuned-conll03-english"
)

config.json:   0%|          | 0.00/979 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  436MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-ca-ner
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/305k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

While you're waiting for the model to download, go and look at it's model card in Hugging Face.


#### BBC example

On 09 Dec 24 the BBC reported
```python
"""
Shares in US chocolate maker Hershey have jumped by more than 10% after
a report that Mondelez International, which owns UK-based Cadbury,
has approached the firm about a potential buyout.
"""
```
[source](https://www.bbc.com/news/articles/c62w62vydnvo)

#### Let's test our NER pipeline on this sample text


In [5]:
# load sample news text
business_news = """
Shares in US chocolate maker Hershey have jumped by more than 10% after
a report that Mondelez International, which owns UK-based Cadbury,
has approached the firm about a potential buyout.
"""

In [6]:
# run pipeline and review the results
results = ner_pipeline(business_news)
results

[{'entity': 'I-LOC',
  'score': np.float32(0.9956012),
  'index': 5,
  'word': 'US',
  'start': 11,
  'end': 13},
 {'entity': 'I-ORG',
  'score': np.float32(0.99883765),
  'index': 8,
  'word': 'Her',
  'start': 30,
  'end': 33},
 {'entity': 'I-ORG',
  'score': np.float32(0.99116665),
  'index': 9,
  'word': '##she',
  'start': 33,
  'end': 36},
 {'entity': 'I-ORG',
  'score': np.float32(0.998396),
  'index': 10,
  'word': '##y',
  'start': 36,
  'end': 37},
 {'entity': 'I-ORG',
  'score': np.float32(0.9993849),
  'index': 22,
  'word': 'Mon',
  'start': 88,
  'end': 91},
 {'entity': 'I-ORG',
  'score': np.float32(0.9881896),
  'index': 23,
  'word': '##del',
  'start': 91,
  'end': 94},
 {'entity': 'I-ORG',
  'score': np.float32(0.99932015),
  'index': 24,
  'word': '##ez',
  'start': 94,
  'end': 96},
 {'entity': 'I-ORG',
  'score': np.float32(0.9994832),
  'index': 25,
  'word': 'International',
  'start': 97,
  'end': 110},
 {'entity': 'I-MISC',
  'score': np.float32(0.99451715),
 

In [7]:
# print the entities
for entity in results:
    print(f"Entity: {entity['entity']}, Value: {entity['word']}")

Entity: I-LOC, Value: US
Entity: I-ORG, Value: Her
Entity: I-ORG, Value: ##she
Entity: I-ORG, Value: ##y
Entity: I-ORG, Value: Mon
Entity: I-ORG, Value: ##del
Entity: I-ORG, Value: ##ez
Entity: I-ORG, Value: International
Entity: I-MISC, Value: UK
Entity: I-ORG, Value: C
Entity: I-ORG, Value: ##ad
Entity: I-ORG, Value: ##bury


In [ ]:
# initalize the NER pipeline
ner_pipeline = pipeline(
  task="token-classification",
  model="CAMeL-Lab/bert-base-arabic-camelbert-ca-ner"
)

In [ ]:
business_news_ar = """
ارتفعت أسهم شركة هيرشي الأمريكية لصناعة الشوكولاتة بأكثر من 10% بعد تقرير يفيد بأن شركة مونديليز إنترناشونال، التي تمتلك شركة كادبوري البريطانية، قد تواصلت مع الشركة بشأن عملية استحواذ محتملة.
"""

In [ ]:
# run pipeline and review the results
results = ner_pipeline(business_news_ar)
results

[{'entity': 'B-ORG',
  'score': np.float32(0.9280191),
  'index': 4,
  'word': 'هير',
  'start': 18,
  'end': 21},
 {'entity': 'I-ORG',
  'score': np.float32(0.9247997),
  'index': 5,
  'word': '##شي',
  'start': 21,
  'end': 23},
 {'entity': 'B-ORG',
  'score': np.float32(0.92792886),
  'index': 20,
  'word': 'مون',
  'start': 89,
  'end': 92},
 {'entity': 'I-ORG',
  'score': np.float32(0.97625947),
  'index': 21,
  'word': '##ديل',
  'start': 92,
  'end': 95},
 {'entity': 'I-ORG',
  'score': np.float32(0.9852255),
  'index': 22,
  'word': '##يز',
  'start': 95,
  'end': 97},
 {'entity': 'I-ORG',
  'score': np.float32(0.97478306),
  'index': 23,
  'word': 'إنت',
  'start': 98,
  'end': 101},
 {'entity': 'I-ORG',
  'score': np.float32(0.966155),
  'index': 24,
  'word': '##رنا',
  'start': 101,
  'end': 104},
 {'entity': 'I-ORG',
  'score': np.float32(0.93330306),
  'index': 25,
  'word': '##شون',
  'start': 104,
  'end': 107},
 {'entity': 'I-ORG',
  'score': np.float32(0.9039618),
  '

In [14]:
# print the entities
for entity in results:
    print(f"Entity: {entity['entity']}, Value: {entity['word']}")

Entity: B-ORG, Value: هير
Entity: I-ORG, Value: ##شي
Entity: B-ORG, Value: مون
Entity: I-ORG, Value: ##ديل
Entity: I-ORG, Value: ##يز
Entity: I-ORG, Value: إنت
Entity: I-ORG, Value: ##رنا
Entity: I-ORG, Value: ##شون
Entity: I-ORG, Value: ##ال
Entity: B-ORG, Value: كاد
Entity: I-ORG, Value: ##بور
Entity: I-ORG, Value: ##ي


#### Results
- Notice that Hershey, Mondelez International, and Cadbury are all correctly identified as organizations.
- Each of them are split into subwords. For example, Hershey is split into three tokens: "her", "##she",and "##y".
  - This is because the model uses a technique called WordPiece tokenization, which splits words into subwords when necessary. This allows the model to handle out-of-vocabulary words and improve generalization.
- In contrast US is correctly identified as a location, and UK is identified as a miscellaneous entity.


# Let's modify our pipeline to use whole words
There are several ways to do this, but one of the easiest is to change the aggregation strategy to simple

```python
ner_pipeline = pipeline("ner", aggregation_strategy="simple")
```


aggregation_strategy="simple"


In [15]:
# initalize the NER pipeline with aggregation_strategy set to simple
ner_simple = pipeline("ner", aggregation_strategy="simple")

[transformers] No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
ner_simple(business_news)

[{'entity_group': 'LOC',
  'score': np.float32(0.9956012),
  'word': 'US',
  'start': 11,
  'end': 13},
 {'entity_group': 'ORG',
  'score': np.float32(0.9961334),
  'word': 'Hershey',
  'start': 30,
  'end': 37},
 {'entity_group': 'ORG',
  'score': np.float32(0.9965944),
  'word': 'Mondelez International',
  'start': 88,
  'end': 110},
 {'entity_group': 'MISC',
  'score': np.float32(0.99451715),
  'word': 'UK',
  'start': 123,
  'end': 125},
 {'entity_group': 'ORG',
  'score': np.float32(0.9962928),
  'word': 'Cadbury',
  'start': 132,
  'end': 139}]

#### Simple Aggregation Strategy results
Notice that now we have a single entity for each word in the text. This is because the simple aggregation strategy returns each word as a separate entity. Notice that we get "LOC", "ORG", "MISC" instead of B/I-LOC


# Try It!
- Run the `ner_pipeline` pipeline on our `reviews.csv` and see what entities are detected in the reviews.
- Run the `ner_simple` pipeline on the reviews list and see what entities are detected in the reviews.
- Compare the results of the two pipelines. What differences do you notice?
- How well did our model do at detecting new entities in the reviews?


In [17]:
# run the ner_pipeline we created on the reviews list and see what entities are detected in the reviews.
ner_pipeline(reviews)

[[{'entity': 'I-ORG',
   'score': np.float32(0.4042618),
   'index': 2,
   'word': 'ab',
   'start': 2,
   'end': 4},
  {'entity': 'I-ORG',
   'score': np.float32(0.42757162),
   'index': 3,
   'word': '##s',
   'start': 4,
   'end': 5},
  {'entity': 'I-ORG',
   'score': np.float32(0.47863775),
   'index': 4,
   'word': '##ol',
   'start': 5,
   'end': 7},
  {'entity': 'I-ORG',
   'score': np.float32(0.43942016),
   'index': 5,
   'word': '##ut',
   'start': 7,
   'end': 9},
  {'entity': 'I-ORG',
   'score': np.float32(0.31824586),
   'index': 9,
   'word': '##ove',
   'start': 14,
   'end': 17},
  {'entity': 'I-ORG',
   'score': np.float32(0.5060187),
   'index': 10,
   'word': 'the',
   'start': 18,
   'end': 21},
  {'entity': 'I-ORG',
   'score': np.float32(0.5364266),
   'index': 11,
   'word': 'Te',
   'start': 22,
   'end': 24},
  {'entity': 'I-ORG',
   'score': np.float32(0.43020254),
   'index': 12,
   'word': '##ch',
   'start': 24,
   'end': 26},
  {'entity': 'I-ORG',
   'sco

In [18]:
# run the ner_simple pipeline on the reviews list and see what entities are detected in the reviews.
ner_simple(reviews)

[[{'entity_group': 'MISC',
   'score': np.float32(0.96840924),
   'word': 'TechWave X1',
   'start': 22,
   'end': 33}],
 [{'entity_group': 'MISC',
   'score': np.float32(0.984529),
   'word': 'TechWave X1',
   'start': 32,
   'end': 43}],
 [{'entity_group': 'MISC',
   'score': np.float32(0.97012347),
   'word': 'TechWave X1',
   'start': 4,
   'end': 15}],
 [{'entity_group': 'MISC',
   'score': np.float32(0.97637796),
   'word': 'TechWave X1',
   'start': 12,
   'end': 23}],
 [{'entity_group': 'MISC',
   'score': np.float32(0.9828328),
   'word': 'TechWave X1',
   'start': 4,
   'end': 15}],
 [{'entity_group': 'MISC',
   'score': np.float32(0.96720695),
   'word': 'TechWave X1',
   'start': 4,
   'end': 15}]]

- Compare the results of the two pipelines. What differences do you notice?
- How well did our model do at detecting new entities in the reviews?


A: Both models did well at detecting our new product as entity and labeling it as MISC. The simple model returned one entity group per review instead of subwords.


#### BONUS: Try switching to a named model from the Hugging Face hub!


In [ ]:
# new_ner = pipeline("ner", model="jean-baptiste/roberta-large-ner-english", aggregation_strategy="simple")
# or new_ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")

In [ ]:
# new_ner(reviews)

## Conclusion
- Named Entity Recognition can be a powerful tool for finding/tracking people, places, and things (such as companies) in text
- This has many useful applications in business and finance (tracking company news, regulatory changes, etc.), as well as biotech (finding drugs, genes, etc.)
- This model is powerful when it is paired with other models (for example tracking sentiment by entity)
